In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR
import os
from pathlib import Path
import matplotlib.pyplot as plt

save_path = "CES_pkl_files"
data = {}
for file in os.listdir(save_path):
    if file.endswith(".pkl"):
        name = file[:-4]
        data[name] = pd.read_pickle(os.path.join(save_path,file))
        print(f"Loaded {name}")

omega = data["omega"]
bea_prices = data["bea_prices"]
bea_expenditures = data["bea_expenditures"]
bea_quantities = data["bea_quantities"]
bea_PQE_merged = data["bea_PQE_merged"]
delta_final = data["delta_final"]

Loaded omega
Loaded bea_prices
Loaded bea_expenditures
Loaded bea_quantities
Loaded bea_PQE_merged
Loaded delta_final


## CES additional data cleaning

Steps 1 - 3 and Basic pre-processing

In [2]:
"""Borrowing Tony's omega data, but change how 0s are cleaned and processed based on CES_Production_Take_Two by Nick"""
"""Is omega in step 2 mean the elements of A, is it related to omega in Tony's work? (Check with Nick)"""

delta_final_num = delta_final.apply(pd.to_numeric, errors="coerce")
A_base = delta_final_num.T   # Ben's convention: columns are buyers, meaning they sum to gamma
gamma = delta_final_num.sum(axis=1) 
print("shape:", A_base.shape)

products = A_base.columns
price_panel = (bea_prices.pivot_table(index="date", columns="products", values="Prices", aggfunc="first").sort_index())
price_panel = price_panel.reindex(columns = products)
missing = set(products) - set(bea_prices["products"].unique())
print(len(missing), sorted(missing)[:10])

A_base


shape: (210, 210)
0 []


,accessories and parts,air transportation,alcohol in purchased meals,all other professional medical services,"amusement parks, campgrounds, and related recreational services","audio discs, tapes, vinyl, and permanent digital downloads",audio equipment,auto leasing,bakery products,beef and veal,...,video and audio streaming and rental,"video discs, tapes, and permanent digital downloads",vocational rehabilitation services,watches,water supply and sewage maintenance,water transportation,window coverings,wine,women's and girls' clothing,workers' compensation
accessories and parts,0.098140,0.000800,0.001887,0.001560,0.003495,0.004488,0.008639,0.007223,0.005449,0.000845,...,0.004422,0.001818,0.011634,0.004941,0.003393,0.003765,0.004972,0.015942,0.002210,0.000579
air transportation,0.000500,0.000064,0.000156,0.000133,0.000277,0.000362,0.000505,0.000410,0.000315,0.000045,...,0.000457,0.000202,0.000233,0.000217,0.000179,0.000403,0.000170,0.000451,0.000098,0.000138
alcohol in purchased meals,0.003793,0.001081,0.001720,0.002704,0.001731,0.001713,0.002710,0.001883,0.002230,0.000335,...,0.004329,0.001581,0.003714,0.001158,0.002084,0.001152,0.001086,0.002823,0.000651,0.000660
all other professional medical services,0.000460,0.000050,0.000082,0.000216,0.000159,0.000214,0.000296,0.000113,0.000182,0.000040,...,0.000181,0.000128,0.000109,0.000149,0.000329,0.000122,0.000115,0.000390,0.000082,0.000118
"amusement parks, campgrounds, and related recreational services",0.002301,0.003404,0.001537,0.000503,0.001739,0.001845,0.002000,0.001478,0.001366,0.000461,...,0.001939,0.001625,0.000531,0.000632,0.000930,0.003611,0.000662,0.002385,0.000655,0.000576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
water transportation,0.001158,0.003968,0.000607,0.000187,0.000529,0.000884,0.001062,0.000386,0.000749,0.000114,...,0.000795,0.000371,0.000391,0.000581,0.000702,0.006410,0.000878,0.001539,0.000342,0.000072
window coverings,0.000750,0.000033,0.000186,0.000119,0.000202,0.000214,0.000417,0.000151,0.000326,0.000051,...,0.000188,0.000088,0.000545,0.000217,0.000134,0.000111,0.000563,0.000625,0.000098,0.000050
wine,0.000525,0.000161,0.001293,0.000081,0.000414,0.000096,0.000222,0.000040,0.000106,0.000021,...,0.000059,0.000035,0.000046,0.000060,0.000091,0.000062,0.000056,0.034677,0.000053,0.000033
women's and girls' clothing,0.000926,0.000055,0.000099,0.000125,0.000157,0.000201,0.000315,0.000103,0.000261,0.000052,...,0.000105,0.000156,0.000282,0.000153,0.000205,0.000106,0.000799,0.000661,0.000668,0.000016


In [3]:
price_panel.dtypes

accessories and parts                                              float64
air transportation                                                 float64
alcohol in purchased meals                                         float64
all other professional medical services                            float64
amusement parks, campgrounds, and related recreational services    float64
                                                                    ...   
water transportation                                               float64
window coverings                                                   float64
wine                                                               float64
women's and girls' clothing                                        float64
workers' compensation                                              float64
Length: 210, dtype: object

Check max col sum between 0 and 1

In [4]:
assert (A_base.index == A_base.columns).all()

col_sums = A_base.sum(axis=0)
print(col_sums.mean(), col_sums.max())
assert 0 < col_sums.max() < 1
assert np.allclose(col_sums.to_numpy(), gamma.reindex(products).to_numpy())

0.18373477246765915 0.6467508822111966


Replace non-existing prices

In [5]:
A_monthly = {}
for tm1, t in zip(price_panel.index[:-1], price_panel.index[1:]):
    exists_tm1 = price_panel.loc[tm1].gt(0).to_numpy()  
    existence_mask = np.outer(exists_tm1, exists_tm1) 
    A_monthly[t] = A_base.mul(existence_mask) # Starts from 1959-02-01, careful when referencing

## Production Function Derivation

4-5. Calculate Sales

In [6]:
# does BEA expenditure match PCE at all? - yes, it seems it has the same number of unique products/categories fortunately

sales_monthly = {}

for date in A_monthly.keys():
    pce_expenditure_t = (bea_expenditures.loc[bea_expenditures["date"] == date].set_index("products")["Expenditures"])
    A_t = A_monthly[date]
    I = np.eye(len(A_t))
    pce_expenditure_t = pce_expenditure_t.reindex(A_t.index, fill_value=0) 
    assert pce_expenditure_t.index.equals(A_t.index)

    sales_t = np.linalg.solve(I - A_t.to_numpy(), pce_expenditure_t.to_numpy()) # NS: removed .T
    # To get around Det = 0, we solve for "[I - A_t.T.to_numpy()] @ sales_t = PC_i,t" instead

    sales_monthly[date] = pd.Series(sales_t, index = A_t.index, name = date)

In [7]:
broke = False
for date in sales_monthly:
    bad = sales_monthly[date] < 0
    if bad.any():
        print(date)
        print(sales_monthly[date][bad])
        broke = True
        break
# NS: Sorry, I like confirmations for and against :)
if broke == False:
    print('If you see this, sales are all positive')

If you see this, sales are all positive


6-8. Calculate B

In [8]:
epsilon = 0.3 # arbitrary number

pi_ela_monthly = {} 
pi_gross_monthly = {}

# Need to adjust price panel before calculating inflation, otherwise we run into divide by 0 NaN
# Again, Tony already adjusted data such that missing datapoints are set to 0
for tm1, t in zip(price_panel.index[:-1], price_panel.index[1:]):
    p_tm1 = price_panel.loc[tm1].copy()
    p_t = price_panel.loc[t].copy()

    exit = (p_tm1 != 0) & (p_t == 0)
    p_t.loc[exit] = p_tm1.loc[exit]

    enter = (p_tm1 == 0) & (p_t != 0)
    p_tm1.loc[enter] = p_t[enter]

    pi_t = (p_t / p_tm1) 

    # 0/0 = nan
    missing_both = pi_t.isna()
    pi_t[missing_both] = 1.0

    pi_ela_monthly[t] = pi_t ** (1 - epsilon)
    pi_gross_monthly[t] = pi_t

In [9]:
# double check that the 0/0 nan is caught correctly
# pi_check = {}
# for tm1, t in zip(price_panel.index[:-1], price_panel.index[1:]):
#     p_tm1 = price_panel.loc[tm1].copy()
#     p_t = price_panel.loc[t].copy()

#     exit = (p_tm1 != 0) & (p_t == 0)
#     p_t.loc[exit] = p_tm1.loc[exit]
    
#     enter = (p_tm1 == 0) & (p_t != 0)
#     p_tm1.loc[enter] = p_t[enter]

#     missing_both = (p_tm1 == 0) & (p_t == 0) 
#     p_tm1[missing_both],p_t[missing_both] = 1.0, 1.0

#     pi_t = (p_t / p_tm1) ** (1 - epsilon)
#     pi_check[t] = pi_t


# assert pi_check.keys() == pi_ela_monthly.keys()
# for date in pi_check:
#     assert pi_check[date].equals(pi_ela_monthly[date])

In [10]:
B_monthly = {}

for date in A_monthly.keys():
    A_t = A_monthly[date]
    I = np.eye(len(A_t))
 
    # separated so I don't get too dizzy (NS: sensible)
    Leontief_inv = I - A_t.T.to_numpy()     # NS: Added .T
    
    col_sums = A_t.to_numpy().sum(axis=0) 
    diag_matrix = np.diag(1 - col_sums)

    B_t = np.linalg.solve(Leontief_inv, diag_matrix)
    B_monthly[date] = B_t

B_monthly[next(iter(B_monthly))]




array([[5.84696807e-01, 7.86945256e-04, 5.24012431e-03, ...,
        3.49900812e-04, 1.15394946e-03, 9.63428092e-04],
       [9.06626234e-04, 9.38575629e-01, 1.11513081e-03, ...,
        9.12062810e-05, 8.54886129e-05, 1.37287296e-04],
       [1.90868801e-03, 2.54022902e-04, 8.58686447e-01, ...,
        6.72650572e-04, 1.55787819e-04, 6.22801205e-04],
       ...,
       [1.25389656e-02, 7.57719337e-04, 4.55711044e-03, ...,
        5.07100659e-01, 8.64041625e-04, 1.82269967e-03],
       [1.71440118e-03, 1.40895146e-04, 8.65260084e-04, ...,
        3.62138525e-05, 9.28711309e-01, 2.40151710e-04],
       [1.05814258e-03, 1.68558486e-04, 7.99010118e-04, ...,
        2.66880042e-05, 4.33251015e-05, 9.29648200e-01]], shape=(210, 210))

9. Check B is valid (row sums to 1)

In [11]:
broke = False
for date in B_monthly:
    row_sums = B_monthly[date] @ np.ones(len(B_monthly[date]))
    col_sums = B_monthly[date].sum(axis=0)
    
    bad = abs((row_sums) -1) > 0.00001 # all row sums positive so Abs is ok
    if bad.any():
        print("B_matrix stats for", date)
        print("Min row sum:", row_sums.min(), "\nMax row sum:", row_sums.max())
        print("Offending rows:")
        print(B_monthly[date][bad])
        broke = True
        break
if broke == False:
    print('If you see this, B rows sum to 1!')

If you see this, B rows sum to 1!


10 - 12. Calculate Value added inflation

In [12]:
pi_VA_monthly = {}

for t in B_monthly.keys():
    pi_ela_t = pi_ela_monthly[t]
    pi_VA_monthly[t] = np.linalg.solve(B_monthly[t], pi_ela_monthly[t])


In [13]:
# step 11 double checking
assert B_monthly.keys() == pi_VA_monthly.keys()
assert B_monthly.keys() == pi_ela_monthly.keys()
for date in B_monthly.keys():
    assert np.allclose(B_monthly[date] @ pi_VA_monthly[date], pi_ela_monthly[date])

Net Value Added Inflation

In [14]:
net_pi_VA_monthly_dict = {}

for time, value in pi_VA_monthly.items():
    net_pi_VA_monthly_dict[time] = pi_VA_monthly[time]**(1/(1-epsilon)) -1
# pi_VA_monthly no longer has negative values by using delta_final, so omega in CES_take three is different from Tony's?


# I totally forgot to use dataframes but maybe I just stick with the dictionaries at this point?
net_VA_monthly = pd.DataFrame(net_pi_VA_monthly_dict, index = products)
net_VA_monthly = net_VA_monthly.T # flip so that rows = dates, cols = products

net_VA_monthly

,accessories and parts,air transportation,alcohol in purchased meals,all other professional medical services,"amusement parks, campgrounds, and related recreational services","audio discs, tapes, vinyl, and permanent digital downloads",audio equipment,auto leasing,bakery products,beef and veal,...,video and audio streaming and rental,"video discs, tapes, and permanent digital downloads",vocational rehabilitation services,watches,water supply and sewage maintenance,water transportation,window coverings,wine,women's and girls' clothing,workers' compensation
1959-02-01,-0.003318,-0.000002,0.011261,0.000000,-0.002245,0.012000,0.001762,0.000000,-0.003969,0.011191,...,0.000000,0.000000,0.000000,0.002453,0.004524,-0.063400,-0.003047,0.001360,0.000061,0.000251
1959-03-01,0.003259,-0.004778,0.007237,0.000000,0.004672,-0.005833,0.000517,0.000000,0.007765,-0.000177,...,0.000000,0.000000,0.000000,-0.002588,-0.000005,0.036960,-0.003780,0.003482,-0.000256,-0.001164
1959-04-01,0.002291,0.004308,0.014814,0.000000,0.008880,-0.003899,-0.001206,0.000000,0.003587,-0.004231,...,0.000000,0.000000,0.000000,0.001970,0.004358,-0.013029,0.015495,0.003675,0.001664,-0.000187
1959-05-01,0.000963,0.004351,0.015495,0.000000,-0.003008,0.005939,-0.000724,0.000000,0.004290,0.002517,...,0.000000,0.000000,0.000000,0.000367,0.005265,0.035487,0.009656,0.006114,0.000279,0.001266
1959-06-01,-0.007771,0.004249,-0.005510,0.000000,0.009621,-0.000348,-0.004923,0.000000,-0.001876,-0.000624,...,0.000000,0.000000,0.000000,-0.001224,0.003039,-0.057509,0.011702,-0.003049,-0.000735,0.000663
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-05-01,0.011707,-0.016199,0.001034,-0.000004,0.003520,0.001931,0.004543,0.011575,0.007895,-0.003920,...,-0.039755,0.000032,0.001260,-0.021298,0.003042,-0.015434,-0.037958,0.003212,-0.007279,0.003699
2024-06-01,-0.009119,-0.017777,0.003479,0.004439,0.000983,-0.006168,-0.007445,-0.065367,0.003669,-0.002351,...,0.000266,-0.013483,0.004485,0.015759,0.001234,0.040824,0.033822,-0.007488,0.005147,0.014908
2024-07-01,0.015118,0.004415,0.000279,0.000866,0.003480,0.012343,0.013816,0.010761,-0.004351,0.013893,...,0.079417,-0.011353,-0.003646,-0.010574,0.005327,-0.011737,-0.030455,-0.008456,-0.004653,-0.005808
2024-08-01,0.005936,0.004060,0.002526,-0.000471,-0.000881,0.007302,-0.009508,0.020550,-0.002671,0.002225,...,-0.005661,0.006410,0.010374,-0.016039,0.001574,-0.024903,0.049053,0.003756,0.016397,-0.001252


In [15]:
# shouldn't have NaNs but a quick check either way
n_nan_VA = net_VA_monthly.isna().sum().sum()
print(n_nan_VA)

0


In [16]:
# also a check here since we need to use sales_df
sales_df = pd.DataFrame(sales_monthly).T
sales_df = sales_df.reindex(columns=products)

print(sales_df.columns[:5])
print(sales_df.isna().all().sum())

Index(['accessories and parts', 'air transportation',
       'alcohol in purchased meals', 'all other professional medical services',
       'amusement parks, campgrounds, and related recreational services'],
      dtype='object')
0


Clean up sales_df and net_VA_monthly before doing the VAR

In [17]:
sales_positive = sales_df.where(sales_df > 0)

log_sales = np.log(sales_df.where(sales_df > 0))
log_sales = log_sales.ffill()
sales_growth = log_sales.diff() #net_VA_monthly is inflation so it's already a growth rate

# should always pass, but it's here now
common_dates = net_VA_monthly.index.intersection(sales_growth.index)
net_VA_monthly = net_VA_monthly.loc[common_dates]
sales_growth = sales_growth.loc[common_dates]
print("missing dates:", len(net_VA_monthly.index) - len(common_dates))



missing dates: 0


12. VAR

In [18]:
lags = 12 # Tony did 4 quarters, I'll use 12 months for now? doublecheck with Nick on this ***
min_obs = 3 * lags

residual_frames = []
skipped = {}

for p in products:

    # all months for one product
    data_p = pd.DataFrame({"va_inflation": net_VA_monthly[p], "sales_growth": sales_growth[p]})
    data_p = data_p.replace([np.inf, -np.inf], np.nan)
    data_p = data_p.dropna()

    if len(data_p) < min_obs:
        skipped[p] = "not enough data points"

    if (data_p.std() == 0).any():
        skipped[p] = "constant series"

    model_p = VAR(data_p)
    result_p = model_p.fit(lags)

    resid_p = result_p.resid.copy()
    resid_p = resid_p.rename(columns={
        "va_inflation": "residual_va",
        "sales_growth": "residual_sales",
    })
    resid_p["product"] = p
    residual_frames.append(resid_p) 

CES_residuals = pd.concat(residual_frames)
CES_residuals = CES_residuals.rename_axis("date")
CES_residuals = CES_residuals.reset_index()
    

/home/leco/.local/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/leco/.local/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/leco/.local/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/leco/.local/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/home/leco/.local/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency

13. Classify demand and supply shocks

In [19]:
# same sign = demand shock, opposite = supply shock

residual_product = (CES_residuals["residual_va"] * CES_residuals["residual_sales"])

CES_residuals["D_CES"] = (residual_product >= 0).astype(int)

D_CES = CES_residuals.pivot(index = "date", columns = "product", values = "D_CES",)

In [20]:
print("total products:", CES_residuals["product"].nunique)
print("products skiped:", len(skipped))

total products: <bound method IndexOpsMixin.nunique of 0         accessories and parts
1         accessories and parts
2         accessories and parts
3         accessories and parts
4         accessories and parts
                  ...          
153137    workers' compensation
153138    workers' compensation
153139    workers' compensation
153140    workers' compensation
153141    workers' compensation
Name: product, Length: 153142, dtype: object>
products skiped: 0


In [21]:
share_demand = CES_residuals["D_CES"].mean()
print("share classified demand:", round(share_demand, 3))

print("D_CES shape:", D_CES.shape)
print("Net_VA_inflation shape:", net_VA_monthly.shape)

share classified demand: 0.552
D_CES shape: (775, 210)
Net_VA_inflation shape: (788, 210)


14. Calculate S_t

In [22]:
S_monthly = {} # need to be careful about orientation, currently rows = products, col = dates

for t in pi_ela_monthly.keys():
    S_t = (pi_gross_monthly[t] - 1)  / (pi_ela_monthly[t] - 1)
    
    # another divide by 0 case because we set pi_ela_monthly to 1 for so many data points D:
    # placeholder , code can handle some changes
    missing_value = S_t.isna()
    S_t[missing_value] = 0.0 

    S_monthly[t] = S_t
    
list(S_monthly.keys())[:5] #everything else is a dictionary anyways so I guess I stick with it

[Timestamp('1959-02-01 00:00:00'),
 Timestamp('1959-03-01 00:00:00'),
 Timestamp('1959-04-01 00:00:00'),
 Timestamp('1959-05-01 00:00:00'),
 Timestamp('1959-06-01 00:00:00')]

15. Calculate pi_i,t

In [23]:
# LHS = pi_i,t = P_i,t / P_i,t-1 = pi_gross_monthly 
# RHS = S_t * (pi_ela_monthly - 1)
for date in pi_gross_monthly.keys():
    assert np.allclose(pi_gross_monthly[date] - 1, S_monthly[date] * (pi_ela_monthly[date] - 1))
    # The equation does not hold with gross inflation, only net inflation with the extra -1 ***
    # If the next step works without problem I will continue with it then
    # clarify differene between pi in step 12 and 15, 16 ***


    # testing_difference = (pi_gross_monthly[date] - 1) - (S_monthly[date] * (pi_ela_monthly[date] - 1))
    # bad = abs(testing_difference) > 0.0001 
    # if bad.any():
    #     print(testing_difference[bad])
    #     break

16. pi_CPI,t

In [24]:
# caulate omega_CPI_monthly, same way as we did for bea prices
expend_panel = bea_expenditures.pivot_table(index="date", columns="products", values="Expenditures",    aggfunc="first",).sort_index()
expend_panel = expend_panel.reindex(columns=products)

missing = set(products) - set(bea_prices["products"].unique())
print(expend_panel.isna().sum().sum())
print(len(missing), sorted(missing)[:5])

monthly_totals = expend_panel.sum(axis=1)

omega_PCE_monthly = {}

for t in pi_gross_monthly.keys(): # the main operaiton for omega_PCE_monthly is to multiply with pi_gross_monthly, so they should match 
    omega_PCE_monthly[t] = expend_panel.loc[t] / expend_panel.loc[t].sum()

omega_PCE_monthly[next(iter(omega_PCE_monthly))]

0
0 []


accessories and parts                                              0.002843
air transportation                                                 0.001849
alcohol in purchased meals                                         0.014015
all other professional medical services                            0.000000
amusement parks, campgrounds, and related recreational services    0.002494
                                                                     ...   
water transportation                                               0.000003
window coverings                                                   0.002758
wine                                                               0.001829
women's and girls' clothing                                        0.042456
workers' compensation                                              0.001394
Name: 1959-02-01 00:00:00, Length: 210, dtype: float64

17. Weights omega_VA,t

In [25]:
pi_CPI_monthly = {}

for t in pi_gross_monthly.keys():
    pi_CPI_monthly[t] = (omega_PCE_monthly[t] * (pi_gross_monthly[t] - 1)).sum()
    pi_CPI_check = omega_PCE_monthly[t].T @ (pi_gross_monthly[t] - 1)
    assert np.isclose(pi_CPI_monthly[t], pi_CPI_check)

In [26]:
omega_VA_monthly = {}

for t in pi_CPI_monthly.keys():
    A_t = A_monthly[t]
    I = np.eye(len(A_t))

    # not sure about how to do this one exactly ***
    # x' = w'PCE(I - A't)^-1 --> (I - A_t) x = w_PCE --> solve for x with linalg.solve
    # w_VA = xy, where y = diag matrix
    x = np.linalg.solve(I - A_t, omega_PCE_monthly[t])
    col_sums = A_t.to_numpy().sum(axis=0)
    omega_VA_monthly[t] = x @ np.diag(1 - col_sums)

In [27]:
# quick check on variables greater than 0
for date in omega_VA_monthly:
    bad = omega_VA_monthly[date] < 0 # lots of 0s from products that have not entered yet/exited
    if bad.any():
        print(omega_VA_monthly[date][bad])

18. pi_CPI,t

In [28]:
for t in pi_CPI_monthly.keys():
    pi_CPI_check = omega_VA_monthly[t].T @ (pi_VA_monthly[t] - 1)
    # assert np.isclose(pi_CPI_monthly[t], pi_CPI_check)

    difference_test = pi_CPI_monthly[t] - pi_CPI_check
    bad = abs(difference_test) > 0.0001 
    if bad.any():
        print(difference_test[bad])

    # code is all correct, check tmrw ***


[0.00031115]
[0.00025704]
[0.00062954]
[0.00014398]
[0.00087904]
[0.00063758]
[0.00048137]
[0.00076442]
[0.00078258]
[0.00017959]
[0.00035373]
[-0.00021531]
[0.00026857]
[0.0001745]
[0.00110316]
[0.00031895]
[0.00013095]
[0.00051195]
[0.00052637]
[0.00016944]
[0.00032021]
[0.00121138]
[0.00025203]
[-0.00016219]
[-0.00019855]
[0.00019859]
[0.00012456]
[0.0005983]
[0.00024793]
[0.00024759]
[0.00059325]
[0.00079372]
[0.00034723]
[0.00036668]
[0.00014056]
[0.00010924]
[-0.00026322]
[0.00045935]
[0.00155192]
[-0.00038228]
[0.00015783]
[-0.00013135]
[0.00073806]
[0.00026664]
[-0.0002526]
[0.0004474]
[0.00073556]
[0.0006302]
[0.0004242]
[0.00054861]
[0.00045449]
[0.00051081]
[0.0006561]
[0.00047075]
[0.00014357]
[0.00024378]
[0.00045978]
[0.00043259]
[0.00025737]
[0.00049323]
[0.00016665]
[0.00043201]
[0.00045637]
[0.0003137]
[0.00013581]
[0.00039602]
[0.00045058]
[0.00073866]
[0.00097658]
[0.00027193]
[0.00045705]
[0.00111207]
[0.00033501]
[0.00123727]
[0.00067452]
[0.00106473]
[0.00030949]
